<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 35px; border-radius: 15px; text-align: center; color: white; margin-bottom: 20px; box-shadow: 0 4px 15px rgba(0,0,0,0.2);">
  <h1 style="font-size: 2.5em; margin-bottom: 5px; letter-spacing: 1px;">BERT Fine-Tuning — Complete Guide</h1>
  <h3 style="font-weight: 300; margin-top: 5px; opacity: 0.9;">From Sentiment Analysis to NER to Question Answering</h3>
  <hr style="border: 1px solid rgba(255,255,255,0.3); width: 50%; margin: 15px auto;">
  <p style="font-size: 1.3em; margin-bottom: 0;"><strong>Ayush Singh</strong> &nbsp;|&nbsp; AI Engineer</p>
</div>

---

<div style="background-color: #eef7ff; padding: 20px; border-left: 5px solid #2196F3; border-radius: 8px; margin: 15px 0;">

### Where BERT is Still Used Today

**1. Search & Retrieval (Vector Search, RAG Base Models)**
- For generating dense embeddings (e.g., Sentence-Transformers, MiniLM)
- Stored in FAISS, Pinecone, Milvus for fast similarity search
- Used in smaller LLM pipelines (retriever + generator architecture)

**2. Enterprise-level NLP Tasks (Fast & Cost-Effective)**
- Named Entity Recognition (NER), Sentiment Analysis
- Classification tasks (spam detection, intent classification)
- Summarization using lightweight variants (DistilBERT)

**3. Hybrid Pipelines with LLMs**
- BERT embeddings for the retriever, then an LLM generates the answer (RAG architecture)

**4. Multilingual NLP**
- XLM-R (a multilingual BERT version) — top choice for 100+ languages
- Used for translation and cross-lingual search

**5. On-Device / Low-Latency Inference**
- For mobile apps and edge devices where GPT/Claude can't run
- Quantized DistilBERT/MiniLM models for chatbots and offline NLP tasks

</div>

<div style="background-color: #f9f9f9; padding: 15px; border-radius: 10px; border: 1px solid #ddd;">

### BERT Family — Quick Comparison

| **Model**           | **Main Use-Cases**                                  | **Why Still Used**                    |
| ------------------- | --------------------------------------------------- | ------------------------------------- |
| BERT / DistilBERT   | NER, classification, embeddings                     | Small, fast, cheap inference          |
| RoBERTa / DeBERTa   | Classification, QA, summarization                   | High accuracy, Kaggle/enterprise use  |
| MPNet / MiniLM      | Vector search, semantic retrieval (RAG)             | Best for FAISS/Pinecone retrieval     |
| T5 / Flan-T5        | Summarization, translation, instruction tasks       | Lightweight text-to-text generation   |
| BART / Pegasus      | Abstractive summarization                           | Less resource-hungry than LLMs        |
| XLNet / Electra     | Classification, QA (legacy setups)                  | Still optimized for speed             |
| XLM-R / mT5 / LaBSE | Multilingual NLP, translation, cross-lingual search | 100+ language support, enterprise use |

</div>

<div style="background-color: #e8f5e9; padding: 12px; border-left: 5px solid #4caf50; border-radius: 8px;">

### Setup — Install Required Libraries

We need three key packages:
- **`datasets`** — Hugging Face library for loading benchmark NLP datasets (IMDb, SQuAD, WikiAnn, etc.)
- **`transformers`** — The core library for BERT models, tokenizers, and training utilities
- **`fsspec`** — Filesystem abstraction needed by `datasets` for data caching

</div>

In [ ]:
!pip install -q datasets fsspec transformers

<div style="background-color: #e3f2fd; padding: 12px; border-left: 5px solid #1565c0; border-radius: 8px;">

### Step 1 — Import Libraries

- **`load_dataset`** — Fetches datasets from the Hugging Face Hub
- **`BertTokenizer`** — Converts raw text into token IDs that BERT understands (WordPiece tokenization)
- **`BertForSequenceClassification`** — BERT with a classification head on top (for sentiment, spam, etc.)
- **`Trainer`** — High-level API that handles the entire training loop (forward, backward, optimization)
- **`TrainingArguments`** — Configuration for training (epochs, batch size, learning rate, etc.)

</div>

<div style="background: linear-gradient(135deg, #11998e 0%, #38ef7d 100%); padding: 20px; border-radius: 12px; color: white; margin: 20px 0; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">
  <h1 style="margin: 0;">PART 1 — BERT for Sentiment Analysis (IMDb Reviews)</h1>
  <p style="margin: 5px 0 0 0; opacity: 0.9; font-size: 1.1em;">Fine-tuning BERT on the IMDb Movie Reviews dataset using Hugging Face Trainer API</p>
</div>

<div style="background-color: #e8f5e9; padding: 15px; border-radius: 8px; margin-top: 10px;">

**What we'll do in this section:**
1. Load the **IMDb dataset** (movie review sentiment — positive/negative)
2. **Tokenize** the text using BERT's WordPiece tokenizer
3. **Fine-tune** `bert-base-uncased` for binary classification
4. **Evaluate** the model and make predictions
5. **Push** the model to Hugging Face Hub

</div>

In [ ]:
from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

<div style="background-color: #f1f8e9; padding: 12px; border-left: 5px solid #689f38; border-radius: 8px;">

### Step 2 — Load the IMDb Dataset

**IMDb** contains 50,000 movie reviews (25K train + 25K test), each labeled as **positive** (1) or **negative** (0). It's the gold standard for binary sentiment classification.

</div>

In [ ]:
# Optional heavier datasets for practice:
# load_dataset("ag_news")
# load_dataset("dbpedia_14")

<div style="background-color: #fff8e1; padding: 12px; border-left: 5px solid #f9a825; border-radius: 8px;">

**Subsetting for Speed:** We select only **1000 train** and **500 test** samples to make training fast for this demo. In production, you'd use the full dataset.

</div>

<div style="background-color: #e8eaf6; padding: 12px; border-left: 5px solid #5c6bc0; border-radius: 8px;">

### Step 3 — Initialize the Tokenizer

**`bert-base-uncased`** tokenizer converts text to lowercase and splits words using **WordPiece** algorithm:
- `"unbelievable"` → `["un", "##believ", "##able"]`
- Special tokens added: `[CLS]` at start, `[SEP]` at end

</div>

In [ ]:
# Real-world use-cases for this setup:
# - Customer review classification (positive / negative / neutral)
# - Support ticket routing (billing, technical, general)
# - Email spam or category detection

In [ ]:
from datasets import load_dataset
from transformers import BertTokenizer

# Pull the full IMDb corpus from Hugging Face
raw_data = load_dataset("imdb")

<div style="background-color: #fce4ec; padding: 12px; border-left: 5px solid #e91e63; border-radius: 8px;">

### Step 4 — Preprocessing Pipeline

The `preprocess()` function does **3 things** in one call:
1. **`map(tokenize_fn)`** — Applies tokenization to every sample in batch mode; removes raw `text` column to save memory
2. **`rename_column("label" → "labels")`** — Hugging Face Trainer expects the column to be named `labels` (plural)
3. **`set_format("torch")`** — Converts the dataset to PyTorch tensors so it's ready for the model

</div>

In [ ]:
raw_data

In [ ]:
tr_set = raw_data["train"].select(range(1000))
te_set = raw_data["test"].select(range(500))

<div style="background-color: #e0f7fa; padding: 12px; border-left: 5px solid #00acc1; border-radius: 8px;">

### Step 5 — Load the BERT Model

**`BertForSequenceClassification`** = BERT encoder + a **linear classification head** on top.
- `num_labels=2` → Binary classification (positive vs. negative)
- The pre-trained BERT weights are loaded, but the classification head is **randomly initialized**
- Fine-tuning will update **all** weights (encoder + head)

```
Input → [BERT Encoder (12 layers)] → [CLS] token embedding → [Linear Layer] → 2 logits → Softmax → Prediction
```

</div>

<div style="background-color: #f9fbe7; padding: 12px; border-left: 5px solid #9e9d24; border-radius: 8px;">

**Inspecting BERT's Architecture:** The cell below prints all **12 encoder layers**. Each layer contains:
- **Self-Attention** (query, key, value projections + attention output)
- **Feed-Forward Network** (intermediate dense → GELU → output dense)
- **LayerNorm** + **Dropout** after each sub-layer

</div>

<div style="background-color: #fff3e0; padding: 12px; border-left: 5px solid #ef6c00; border-radius: 8px;">

### Optional — Layer Freezing Strategy

The commented code below shows how to **freeze** BERT's encoder and only train the last 2 layers + classification head. This is useful when:
- You have a **small dataset** (prevents overfitting)
- You want **faster training** (fewer parameters to update)
- You want to preserve BERT's **general language understanding**

**Current setting:** All layers are trainable (full fine-tuning). Uncomment to try partial freezing.

</div>

<div style="background-color: #e8f5e9; padding: 12px; border-left: 5px solid #2e7d32; border-radius: 8px;">

### Step 6 — Training Configuration

| Parameter | Value | What It Does |
|-----------|-------|-------------|
| `output_dir` | `./bert-finetuned-imdb` | Where checkpoints and model files are saved |
| `num_train_epochs` | `1` | Number of full passes through the training data |
| `per_device_train_batch_size` | `8` | Samples processed per GPU in one step |
| `learning_rate` | `2e-5` | The standard BERT fine-tuning LR (recommended: 2e-5 to 5e-5) |
| `weight_decay` | `0.01` | L2 regularization to prevent overfitting |
| `report_to` | `"none"` | Disables WandB/TensorBoard auto-logging |

</div>

<div style="background-color: #e3f2fd; padding: 12px; border-left: 5px solid #1976d2; border-radius: 8px;">

### Step 7 — Initialize the Trainer

The **Trainer** wraps the entire training loop. It receives:
- **`model`** — The BERT model to fine-tune
- **`args`** — Training hyperparameters (defined above)
- **`train_dataset`** — Preprocessed training data (tokenized + formatted as tensors)
- **`eval_dataset`** — Preprocessed test data for evaluation

Behind the scenes, Trainer handles: batching, gradient computation, optimizer steps, logging, and checkpointing.

</div>

<div style="background-color: #f3e5f5; padding: 12px; border-left: 5px solid #8e24aa; border-radius: 8px;">

### Step 8 — Train the Model

`trainer.train()` starts the fine-tuning process. For each batch:
1. **Forward pass** — Input tokens go through BERT → get logits
2. **Loss computation** — Cross-entropy loss between predicted logits and true labels
3. **Backward pass** — Compute gradients via backpropagation
4. **Parameter update** — AdamW optimizer updates model weights

</div>

<div style="background-color: #efebe9; padding: 12px; border-left: 5px solid #795548; border-radius: 8px;">

**TensorBoard** — Visualize training metrics (loss curves, learning rate, etc.) in an interactive dashboard. The `--logdir` flag points to where training logs were saved.

</div>

<div style="background-color: #e0f2f1; padding: 12px; border-left: 5px solid #00897b; border-radius: 8px;">

### Step 9 — Save the Model & Tokenizer

Saving both the **model weights** and **tokenizer** to disk so they can be:
- Reloaded later for inference without retraining
- Shared with others or deployed to production
- Pushed to Hugging Face Hub

</div>

In [ ]:
# Load the WordPiece tokenizer
bert_tok = BertTokenizer.from_pretrained("bert-base-uncased")

<div style="background-color: #fce4ec; padding: 12px; border-left: 5px solid #c62828; border-radius: 8px;">

### Step 10 — Evaluate on Test Set

`trainer.evaluate()` runs the model on the test dataset **without gradients** (inference mode) and returns metrics like **eval_loss**. This tells us how well the model generalizes to unseen data.

</div>

<div style="background-color: #fff3e0; padding: 15px; border-left: 5px solid #ff9800; border-radius: 8px;">

### How Tokenization Works Here

| Parameter | What It Does |
|-----------|-------------|
| `examples["text"]` | The raw review text — each row in the IMDb dataset contains a review |
| `padding="max_length"` | Pads every sentence to the **same length** (up to 256 tokens) with `[PAD]` tokens |
| `truncation=True` | If the text is **longer** than 256 tokens, it gets **cut off** (truncated) |
| `max_length=256` | The fixed sequence length — balances memory usage vs. capturing enough context |

**Why 256?** Full BERT supports up to 512 tokens, but 256 is a good trade-off between speed and information retention for movie reviews.

</div>

In [ ]:
# Encode a batch of raw review strings into token IDs
def encode_batch(batch):
    return bert_tok(batch["text"], padding="max_length", truncation=True, max_length=256)

In [ ]:
# Tokenize → rename column → convert to PyTorch tensors
def build_split(ds):
    ds = ds.map(encode_batch, batched=True, remove_columns=["text"])
    ds = ds.rename_column("label", "labels")
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    return ds

In [ ]:
tr_set = build_split(tr_set)

In [ ]:
te_set = build_split(te_set)

In [ ]:
# Load BERT with a 2-class classification head on top
clf_model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

In [ ]:
for enc_blk in clf_model.bert.encoder.layer:
    print(enc_blk)

In [ ]:
# Partial freeze strategy — uncomment to train only the top 2 layers + classifier head

# for param in clf_model.bert.parameters():
#     param.requires_grad = False  # freeze the whole encoder

# for blk in clf_model.bert.encoder.layer[-2:]:
#     for param in blk.parameters():
#         param.requires_grad = True  # unfreeze last 2 encoder blocks

In [ ]:
from transformers import TrainingArguments

run_cfg = TrainingArguments(
    output_dir="./bert-finetuned-imdb",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    logging_dir="./logs",
    learning_rate=2e-5,
    weight_decay=0.01,
    report_to="none"
)

In [ ]:
# Wire up the Trainer with model, config, and split datasets
hf_trainer = Trainer(
    model=clf_model,
    args=run_cfg,
    train_dataset=tr_set,
    eval_dataset=te_set,
)

In [ ]:
hf_trainer.train()

<div style="background-color: #eceff1; padding: 12px; border-left: 5px solid #607d8b; border-radius: 8px;">

### Alternative Approach — Dynamic Padding (Commented Code)

The code below shows an alternative approach using **`DataCollatorWithPadding`** which pads each batch to the length of its **longest sample** instead of a fixed `max_length`. This is **more memory efficient** because shorter batches use less padding.

**Fixed padding** (Part 1): Every sample padded to 256 tokens — simpler but wastes memory on short reviews.

**Dynamic padding** (below): Each batch padded to its own max — saves memory, especially with varied-length texts.

</div>

In [ ]:
!tensorboard --logdir=./logs

<div style="background-color: #e8eaf6; padding: 12px; border-left: 5px solid #283593; border-radius: 8px;">

### Part 2 — Import All Required Libraries

| Library | Purpose |
|---------|---------|
| `torch`, `torch.nn` | PyTorch core — tensors, neural network layers |
| `Dataset`, `DataLoader` | PyTorch data utilities — custom datasets + batching |
| `BertTokenizerFast` | Fast Rust-backed tokenizer (supports `word_ids()` for NER alignment) |
| `BertForSequenceClassification` | BERT + classification head (sentiment) |
| `BertForTokenClassification` | BERT + per-token classification head (NER) |
| `BertForQuestionAnswering` | BERT + start/end position heads (QA) |
| `get_linear_schedule_with_warmup` | LR scheduler: warmup → linear decay |
| `AdamW` | Adam optimizer with decoupled weight decay |
| `sklearn.metrics` | Accuracy, F1 score, classification report |
| `tqdm` | Progress bars for training loops |

</div>

<div style="background-color: #fff8e1; padding: 12px; border-left: 5px solid #ff8f00; border-radius: 8px;">

**Device Selection:** Automatically uses **GPU (CUDA)** if available, otherwise falls back to **CPU**. GPU accelerates matrix operations by 10-50x, which is critical for transformer training.

</div>

In [ ]:
hf_trainer.save_model("./bert-finetuned-imdb")

In [ ]:
bert_tok.save_pretrained("./bert-finetuned-imdb")

<div style="background: linear-gradient(135deg, #84fab0 0%, #8fd3f4 100%); padding: 15px; border-radius: 10px; margin: 15px 0;">
  <h4 style="margin: 0; color: #333;">Quick Python OOP Refresher — Understanding `__len__` and `__getitem__`</h4>
</div>

<div style="background-color: #e8f5e9; padding: 12px; border-radius: 8px;">

Before building the real Dataset class, here's a simple **Basket** example to understand the two magic methods that PyTorch's `Dataset` requires:

- **`__len__()`** → Returns the total number of items (used by `DataLoader` to know the dataset size)
- **`__getitem__(idx)`** → Returns one item by index (used by `DataLoader` to fetch individual samples)

</div>

In [ ]:
eval_out = hf_trainer.evaluate()

In [ ]:
print(eval_out)

<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 15px; border-radius: 10px; color: white; margin: 15px 0;">
  <h4 style="margin: 0;">TextClassificationDataset — Custom PyTorch Dataset for BERT</h4>
</div>

<div style="background-color: #ede7f6; padding: 12px; border-radius: 8px;">

This class wraps raw text + labels into a format BERT can consume:

| Method | What It Returns |
|--------|----------------|
| `__len__()` | Total number of samples |
| `__getitem__(idx)` | A dictionary with `input_ids`, `attention_mask`, and `labels` tensors for one sample |

**Inside `__getitem__`:**
1. Grab the text and label at index `idx`
2. Tokenize the text (padding + truncation to `max_length`)
3. Flatten tensors from `(1, seq_len)` → `(seq_len,)`
4. Return the dictionary — `DataLoader` will batch these automatically

</div>

<div style="background: linear-gradient(135deg, #f093fb 0%, #f5576c 100%); padding: 18px; border-radius: 12px; color: white; margin: 20px 0;">
  <h2 style="margin: 0;">Prediction — Testing the Fine-Tuned Model</h2>
  <p style="margin: 5px 0 0 0; opacity: 0.9;">Load the saved model and run inference on new, unseen text</p>
</div>

<div style="background-color: #fce4ec; padding: 15px; border-radius: 8px;">

**What happens here:**
- We reload the saved **tokenizer** and **model** from disk (simulating a production environment)
- We use Hugging Face's `pipeline()` API — a **one-line inference** wrapper
- The pipeline handles tokenization, forward pass, and softmax internally

</div>

In [ ]:
saved_tok = BertTokenizer.from_pretrained("/content/bert-finetuned-imdb")
saved_clf = BertForSequenceClassification.from_pretrained("/content/bert-finetuned-imdb")

In [ ]:
from transformers import pipeline
review_pipe = pipeline("text-classification", model=saved_clf, tokenizer=saved_tok)

<div style="background: linear-gradient(135deg, #f093fb 0%, #f5576c 100%); padding: 15px; border-radius: 10px; color: white; margin: 15px 0;">
  <h4 style="margin: 0;">NERDataset — Custom Dataset for Named Entity Recognition</h4>
</div>

<div style="background-color: #fce4ec; padding: 12px; border-radius: 8px;">

NER is trickier than classification because labels are **per-token**, not per-sentence. The key challenge is **subword alignment** — BERT splits words into subwords, but labels are per-word.

**What `__getitem__` does:**
1. Tokenizes the word list with `is_split_into_words=True` (tells BERT these are pre-tokenized words)
2. Gets `word_ids()` — maps each subword token back to its original word index
3. Aligns labels: first subword gets the real label, continuation subwords get `-100`
4. Pads/truncates `aligned_labels` to match `max_length`

</div>

In [ ]:
sample_text = "The cinematography was breathtaking and the story kept me hooked!"
pred_out = review_pipe(sample_text)

In [ ]:
print(pred_out)  # Example output: [{'label': 'POSITIVE', 'score': 0.98}]

<div style="background: linear-gradient(135deg, #ffecd2 0%, #fcb69f 100%); padding: 18px; border-radius: 12px; margin: 20px 0;">
  <h3 style="margin: 0; color: #333;">Pushing to Hugging Face Hub</h3>
  <p style="margin: 5px 0 0 0; color: #555;">Share your fine-tuned model with the world — anyone can download and use it!</p>
</div>

<div style="background-color: #fff8e1; padding: 15px; border-left: 5px solid #ffc107; border-radius: 8px;">

**Steps:**
1. **`notebook_login()`** — Authenticate with your Hugging Face account (requires an access token)
2. **`whoami()`** — Verify you're logged in correctly
3. **`push_to_hub()`** — Upload both the tokenizer and model weights to your HF repository

After pushing, your model will be available at: `https://huggingface.co/your-username/model-name`

</div>

<div style="background: linear-gradient(135deg, #a18cd1 0%, #fbc2eb 100%); padding: 15px; border-radius: 10px; margin: 15px 0;">
  <h4 style="margin: 0; color: #333;">BERTNERClassifier — Full NER Training Pipeline</h4>
</div>

<div style="background-color: #f3e5f5; padding: 12px; border-radius: 8px;">

This class mirrors `BERTTextClassifier` but adapted for **token-level** classification:

| Difference | Text Classification | NER |
|-----------|-------------------|-----|
| **Model** | `BertForSequenceClassification` | `BertForTokenClassification` |
| **Labels** | 1 label per sentence (pos/neg) | 1 label per token (9 NER tags) |
| **Prediction** | `argmax` over 2 classes | `argmax` over 9 classes **per token** |
| **Evaluation** | Only compare sentence-level preds | Skip `-100` tokens, compare valid preds only |
| **Dataset** | WikiAnn (tokens + NER tags) | IMDb (text + sentiment) |

</div>

<div style="background: linear-gradient(135deg, #ffecd2 0%, #fcb69f 100%); padding: 15px; border-radius: 10px; margin: 15px 0;">
  <h4 style="margin: 0; color: #333;">QADataset — Dataset for Extractive Question Answering</h4>
</div>

<div style="background-color: #fff3e0; padding: 12px; border-radius: 8px;">

**Extractive QA** = Find the answer **span** within the given context (not generate new text).

**How it works:**
1. Tokenize `[question] [SEP] [context]` together as a pair
2. Use **`offset_mapping`** to map character positions → token positions
3. Find which tokens correspond to the answer's **start** and **end** character positions
4. Return `start_positions` and `end_positions` as labels

**Example:**
```
Context: "Paris is the capital of France"
Question: "What is the capital of France?"
Answer: "Paris" (starts at char 0, ends at char 5)
→ Model learns: start_token=1, end_token=1
```

The model predicts **two things**: which token the answer **starts** at and which token it **ends** at.

</div>

<div style="background: linear-gradient(135deg, #13547a 0%, #80d0c7 100%); padding: 15px; border-radius: 10px; color: white; margin: 15px 0;">
  <h4 style="margin: 0;">BERTQuestionAnswering — Full QA Training Pipeline</h4>
</div>

<div style="background-color: #e0f2f1; padding: 12px; border-radius: 8px;">

**Key differences from classification:**

| Aspect | Classification | Question Answering |
|--------|---------------|-------------------|
| **Input** | Single text | Question + Context pair |
| **Output** | Class logits | **Start logits** + **End logits** (one per token) |
| **Loss** | CrossEntropy on class | CrossEntropy on start + CrossEntropy on end |
| **Prediction** | `argmax(logits)` | `argmax(start_logits)` & `argmax(end_logits)`, then decode token span |
| **Dataset** | SQuAD (Stanford Question Answering Dataset) | IMDb / WikiAnn |

**`answer_question()` method:**
1. Tokenize question + context
2. Get start and end logits from the model
3. Find the token positions with highest scores
4. Decode those tokens back to text → that's the answer!

</div>

<div style="background: linear-gradient(135deg, #ff9a9e 0%, #fad0c4 100%); padding: 20px; border-radius: 12px; text-align: center; margin: 25px 0; box-shadow: 0 3px 10px rgba(0,0,0,0.1);">
  <h2 style="margin: 0; color: #333;">Demo Functions — Run & Test Each Task</h2>
  <p style="margin: 5px 0 0 0; color: #555;">Each demo loads data, trains the model, evaluates it, and tests on custom examples</p>
</div>

<div style="background-color: #e8f5e9; padding: 10px 15px; border-left: 5px solid #43a047; border-radius: 8px;">

**Demo 1 — Text Classification:** Loads 1000 IMDb samples → trains 1 epoch → evaluates accuracy & F1 → predicts sentiment on 3 custom reviews

</div>

<div style="background-color: #f3e5f5; padding: 10px 15px; border-left: 5px solid #8e24aa; border-radius: 8px;">

**Demo 2 — Named Entity Recognition:** Loads 500 WikiAnn samples → trains 1 epoch → evaluates accuracy & F1 → predicts NER tags on custom sentences like "John Smith works at Google in California"

</div>

<div style="background-color: #e0f2f1; padding: 10px 15px; border-left: 5px solid #00897b; border-radius: 8px;">

**Demo 3 — Question Answering:** Loads 500 SQuAD samples → trains 1 epoch → answers custom questions like "What is the capital of France?" given a context paragraph

</div>

<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 30px; border-radius: 15px; text-align: center; color: white; margin-top: 30px; box-shadow: 0 4px 15px rgba(0,0,0,0.2);">
  <hr style="border: 1px solid rgba(255,255,255,0.3); width: 60%; margin: 0 auto 15px auto;">
  <h2 style="margin: 0;">Notebook Complete!</h2>
  <p style="margin: 10px 0 5px 0; font-size: 1.1em;">You've learned how to fine-tune BERT for 3 major NLP tasks:</p>
  <p style="margin: 0; opacity: 0.9;"><strong>Sentiment Analysis</strong> &nbsp;|&nbsp; <strong>Named Entity Recognition</strong> &nbsp;|&nbsp; <strong>Question Answering</strong></p>
  <hr style="border: 1px solid rgba(255,255,255,0.3); width: 40%; margin: 15px auto;">
  <p style="margin: 0; font-size: 1.2em;"><strong>Ayush Singh</strong> &nbsp;|&nbsp; AI Engineer</p>
</div>

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from huggingface_hub import whoami
print(whoami())

In [ ]:
bert_tok.push_to_hub("sunny199/my-bert-imdb2")

In [ ]:
hf_trainer.push_to_hub("sunny199/my-bert-imdb2")

In [ ]:
# Alternative: dynamic padding using DataCollatorWithPadding
# from datasets import load_dataset
# from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding

# raw_data = load_dataset("imdb")
# tr_set = raw_data["train"].select(range(1000))
# te_set = raw_data["test"].select(range(500))

# bert_tok = BertTokenizer.from_pretrained("bert-base-uncased")

# def encode_batch(batch):
#     return bert_tok(batch["text"], truncation=True, max_length=256)

# def build_split(ds):
#     ds = ds.map(encode_batch, batched=True, remove_columns=["text"])
#     ds = ds.rename_column("label", "labels")
#     ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
#     return ds

# tr_set = build_split(tr_set)
# te_set = build_split(te_set)

# clf_model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

# dyn_collator = DataCollatorWithPadding(tokenizer=bert_tok)

# run_cfg = TrainingArguments(
#     output_dir="./bert-finetuned-imdb",
#     num_train_epochs=1,
#     per_device_train_batch_size=8,
#     per_device_eval_batch_size=8,
#     logging_dir="./logs",
#     logging_steps=50,
#     learning_rate=2e-5,
#     weight_decay=0.01,
#     eval_steps=500,
#     save_steps=500,
#     save_total_limit=1,
#     report_to="none"
# )

# hf_trainer = Trainer(
#     model=clf_model,
#     args=run_cfg,
#     train_dataset=tr_set,
#     eval_dataset=te_set,
#     data_collator=dyn_collator,
# )

# hf_trainer.train()

<div style="background: linear-gradient(135deg, #a18cd1 0%, #fbc2eb 100%); padding: 25px; border-radius: 15px; text-align: center; margin: 30px 0; box-shadow: 0 4px 15px rgba(0,0,0,0.15);">
  <h1 style="margin: 0; color: #333;">PART 2 — BERT for Multiple NLP Tasks</h1>
  <p style="margin: 8px 0 0 0; color: #555; font-size: 1.1em;">Custom PyTorch Training Loop — Text Classification, NER & Question Answering</p>
  <hr style="border: 1px solid rgba(0,0,0,0.1); width: 40%; margin: 12px auto;">
  <p style="margin: 0; color: #666;"><strong>Ayush Singh</strong> | AI Engineer</p>
</div>

<div style="background-color: #f3e5f5; padding: 15px; border-radius: 8px; margin-top: 10px;">

**What's different in Part 2?**
- Instead of using the Hugging Face **Trainer API**, we write a **custom PyTorch training loop**
- This gives us **full control** over the training process (gradients, schedulers, logging)
- We tackle **3 different NLP tasks** with BERT:

| Task | Model Head | Dataset | Output |
|------|-----------|---------|--------|
| **Text Classification** | `BertForSequenceClassification` | IMDb | Positive / Negative |
| **Named Entity Recognition** | `BertForTokenClassification` | WikiAnn | PER, ORG, LOC tags per token |
| **Question Answering** | `BertForQuestionAnswering` | SQuAD | Start & end position of answer |

</div>

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    BertTokenizer,
    BertTokenizerFast,
    BertForSequenceClassification,
    BertForTokenClassification,
    BertForQuestionAnswering,
    get_linear_schedule_with_warmup  # LR warmup then linear decay for stable BERT training
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report, f1_score
import numpy as np
from tqdm import tqdm
from torch.optim import AdamW  # AdamW = Adam + decoupled weight decay

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running on: {device}")

<div style="background-color: #e3f2fd; padding: 15px; border-left: 5px solid #1976D2; border-radius: 8px;">

### Understanding Learning Rate Scheduling

**Example calculation:**
- Dataset size = **800 samples**, `batch_size=8` → 1 epoch = `800/8` = **100 batches**
- `epochs=3` → `total_steps = 100 × 3` = **300 total optimizer updates**
- If warmup = 10% → **30 steps warmup**, remaining **270 steps** = linear LR decay

**Why warmup?** BERT's pre-trained weights are sensitive. A sudden high learning rate can destroy the learned representations. Warmup gradually increases LR from 0, letting the model **adjust gently** before full-speed training.

```
LR ▲
   |    /‾‾‾‾‾‾‾‾‾‾\
   |   /              \
   |  /                \
   | /                  \
   |/____________________\___→ Steps
   0   warmup    decay
```

</div>

<div style="background-color: #fff3e0; padding: 15px; border-left: 5px solid #e65100; border-radius: 8px;">

### When to Use a Custom PyTorch Dataset Class?

There are **two approaches** for feeding data to BERT:

| Approach | When to Use |
|----------|-------------|
| **HF `map()` + `set_format()`** | Data is already in Hugging Face Dataset format (like the IMDB example in Part 1) |
| **Custom `Dataset` class** | You have raw Python lists, need custom preprocessing, or use a non-HF tokenizer |

**In Part 2**, we use a custom `Dataset` class because:
- We work with Python lists (`train_texts`, `train_labels`) directly
- We want fine-grained control over how each sample is tokenized and returned
- This is the standard PyTorch way — essential to know for production code

</div>

In [ ]:
class ItemBox:
    def __init__(self, items):
        self.items = items         # store the list

    def __len__(self):
        return len(self.items)     # total count

    def __getitem__(self, pos):
        return self.items[pos]     # fetch by index

In [ ]:
my_box = ItemBox(["Apple", "Banana", "Mango"])

In [ ]:
print(len(my_box))
print(my_box[0])
print(my_box[2])

In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, reviews, sentiments, tok, seq_len=512):
        self.reviews = reviews
        self.sentiments = sentiments
        self.tok = tok
        self.seq_len = seq_len

    def __len__(self):
        return len(self.reviews)

    def __getitem__(self, pos):
        review = str(self.reviews[pos])
        label = int(self.sentiments[pos])

        enc = self.tok(
            review,
            truncation=True,
            padding='max_length',
            max_length=self.seq_len,
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].flatten(),
            'attention_mask': enc['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

<div style="background: linear-gradient(135deg, #13547a 0%, #80d0c7 100%); padding: 18px; border-radius: 12px; color: white; margin: 20px 0;">
  <h3 style="margin: 0;">BERTTextClassifier — Complete Training Pipeline</h3>
</div>

<div style="background-color: #e0f2f1; padding: 15px; border-radius: 8px;">

**How the class works (step by step):**

| Step | Method | What Happens |
|------|--------|-------------|
| 1 | `__init__()` | Load BERT model + tokenizer, move to GPU/CPU |
| 2 | `load_imdb_data()` | Sample train & test texts with labels from IMDb |
| 3 | `train()` | Create DataLoader → for each epoch & batch: **forward pass → loss → backward → update weights → adjust LR** |
| 4 | `evaluate()` | Run test data (no gradients) → collect predictions → compute **accuracy, F1, classification report** |
| 5 | `predict()` | Tokenize new text → forward pass → **softmax** → return label + confidence |

**Key components inside `train()`:**
- **AdamW** — Adam optimizer with weight decay (prevents overfitting)
- **Linear warmup scheduler** — Gentle LR ramp-up then linear decay
- **Gradient clipping** (`clip_grad_norm_`) — Prevents exploding gradients, keeps training stable

</div>

In [ ]:
class TextSentimentModel:
    """End-to-end BERT pipeline for Sentiment / Text Classification"""

    def __init__(self, base='bert-base-uncased', num_classes=2, seq_len=512):
        self.base = base
        self.num_classes = num_classes
        self.seq_len = seq_len
        self.tok = BertTokenizerFast.from_pretrained(base)
        self.net = BertForSequenceClassification.from_pretrained(base, num_labels=num_classes)
        self.net.to(device)

    def fetch_imdb(self, n=5000):
        """Sample n reviews from the IMDb dataset"""
        print("Loading IMDb dataset...")
        ds = load_dataset("imdb")

        tr_idx = np.random.choice(len(ds['train']), min(n, len(ds['train'])), replace=False)
        te_idx = np.random.choice(len(ds['test']), min(n // 4, len(ds['test'])), replace=False)

        tr_texts = [ds['train'][int(i)]['text'] for i in tr_idx]
        tr_labels = [ds['train'][int(i)]['label'] for i in tr_idx]
        te_texts = [ds['test'][int(i)]['text'] for i in te_idx]
        te_labels = [ds['test'][int(i)]['label'] for i in te_idx]

        print(f"Train: {len(tr_texts)}  |  Test: {len(te_texts)}")
        return tr_texts, tr_labels, te_texts, te_labels

    def fit(self, tr_texts, tr_labels, epochs=1, batch_sz=8, lr=2e-5):
        """Run the training loop"""
        train_ds = SentimentDataset(tr_texts, tr_labels, self.tok, self.seq_len)
        loader = DataLoader(train_ds, batch_size=batch_sz, shuffle=True)

        opt = AdamW(self.net.parameters(), lr=lr, weight_decay=0.01)
        n_steps = len(loader) * epochs
        sched = get_linear_schedule_with_warmup(opt, num_warmup_steps=n_steps // 10,
                                                num_training_steps=n_steps)
        self.net.train()

        for ep in range(epochs):
            running_loss = 0
            bar = tqdm(loader, desc=f'Epoch {ep+1}/{epochs}')
            for batch in bar:
                opt.zero_grad()
                ids = batch['input_ids'].to(device)
                mask = batch['attention_mask'].to(device)
                lbls = batch['labels'].to(device)

                out = self.net(input_ids=ids, attention_mask=mask, labels=lbls)
                loss = out.loss
                running_loss += loss.item()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.net.parameters(), 1.0)
                opt.step()
                sched.step()
                bar.set_postfix({'loss': f'{loss.item():.4f}'})

            print(f'Epoch {ep+1} — avg loss: {running_loss / len(loader):.4f}')

    def score(self, te_texts, te_labels, batch_sz=8):
        """Evaluate on test split"""
        test_ds = SentimentDataset(te_texts, te_labels, self.tok, self.seq_len)
        loader = DataLoader(test_ds, batch_size=batch_sz, shuffle=False)

        self.net.eval()
        all_preds, all_true = [], []

        with torch.no_grad():
            for batch in tqdm(loader, desc='Evaluating'):
                ids = batch['input_ids'].to(device)
                mask = batch['attention_mask'].to(device)
                lbls = batch['labels'].to(device)

                out = self.net(input_ids=ids, attention_mask=mask)
                preds = torch.argmax(out.logits, dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_true.extend(lbls.cpu().numpy())

        acc = accuracy_score(all_true, all_preds)
        f1 = f1_score(all_true, all_preds, average='weighted')
        report = classification_report(all_true, all_preds, target_names=['Negative', 'Positive'])
        return acc, f1, report

    def infer(self, texts):
        """Run inference on a list of strings"""
        self.net.eval()
        preds, probs = [], []

        for txt in texts:
            enc = self.tok(txt, truncation=True, padding='max_length',
                           max_length=self.seq_len, return_tensors='pt')
            ids = enc['input_ids'].to(device)
            mask = enc['attention_mask'].to(device)

            with torch.no_grad():
                out = self.net(input_ids=ids, attention_mask=mask)
                prob = torch.softmax(out.logits, dim=1).cpu().numpy()[0]
                pred = torch.argmax(out.logits, dim=1).cpu().numpy()[0]
                preds.append(pred)
                probs.append(prob)

        return preds, probs

<div style="background: linear-gradient(135deg, #ff9a9e 0%, #fecfef 100%); padding: 18px; border-radius: 12px; margin: 20px 0;">
  <h3 style="margin: 0; color: #333;">Understanding NER Token-Label Alignment</h3>
  <p style="margin: 5px 0 0 0; color: #555;">The trickiest part of NER with BERT — aligning subword tokens back to original word labels</p>
</div>

<div style="background-color: #fce4ec; padding: 15px; border-radius: 8px;">

**The Problem:** BERT uses **WordPiece tokenization**, which splits unknown words into subwords. But NER labels are per-**word**, not per-**subword**.

**Example:**

```
Original words:  John    lives   in    London
NER Labels:      B-PER   O       O     B-LOC
```

**Step 1 — BERT tokenizes:**
```
BERT tokens:  [CLS]  John  lives  in  Lon  ##don  [SEP]  [PAD]...
Word IDs:     None    0     1     2    3     3    None   None...
```

**Step 2 — Align labels using Word IDs:**
```
Token:          [CLS]  John  lives  in   Lon   ##don  [SEP]  [PAD]
Word ID:        None    0     1     2     3      3    None   None
Aligned Label:  -100    1     0     0     2    -100   -100   -100
```

**Key rules:**
- **Special tokens** (`[CLS]`, `[SEP]`, `[PAD]`) → label = **`-100`** (ignored by loss function)
- **First subword** of a word → gets the **actual label**
- **Continuation subwords** (`##don`) → label = **`-100`** (ignored)

**Step 3 — What the model receives:**
```python
{
  'input_ids':      tensor([101, 1001, 2002, 1999, 3001, 3010, 102, 0, 0, 0]),
  'attention_mask': tensor([  1,    1,    1,    1,    1,    1,   1, 0, 0, 0]),
  'labels':         tensor([-100,   1,    0,    0,    2, -100, -100, -100, -100, -100])
}
```

</div>

In [ ]:
import torch
from torch.utils.data import Dataset

class TagDataset(Dataset):
    """Dataset for Named Entity Recognition — handles subword alignment"""

    def __init__(self, word_seqs, tag_seqs, tok, seq_len=512):
        self.word_seqs = word_seqs
        self.tag_seqs = tag_seqs
        self.tok = tok
        self.seq_len = seq_len

    def __len__(self):
        return len(self.word_seqs)

    def __getitem__(self, pos):
        words = self.word_seqs[pos]
        tags = self.tag_seqs[pos]

        enc = self.tok(
            words,
            truncation=True,
            padding='max_length',
            max_length=self.seq_len,
            is_split_into_words=True,
            return_tensors='pt'
        )

        word_ids = enc.word_ids(batch_index=0)
        aligned = []
        prev_wid = None

        for wid in word_ids:
            if wid is None:
                aligned.append(-100)
            elif wid != prev_wid:
                aligned.append(tags[wid] if wid < len(tags) else 0)
            else:
                aligned.append(-100)
            prev_wid = wid

        if len(aligned) < self.seq_len:
            aligned += [-100] * (self.seq_len - len(aligned))
        else:
            aligned = aligned[:self.seq_len]

        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels': torch.tensor(aligned, dtype=torch.long)
        }

<div style="background-color: #f3e5f5; padding: 15px; border-left: 5px solid #9c27b0; border-radius: 8px;">

### NER Label Reference — BIO Tagging Scheme

| **Label** | **Full Form** | **Meaning** | **Example** |
|-----------|---------------|-------------|-------------|
| **O** | Outside | Not an entity (normal word) | "works", "at" |
| **B-PER** | Begin - Person | First word of a person's name | "John" → `B-PER` |
| **I-PER** | Inside - Person | Continuation of a person's name | "Mary Jane" → `Mary=B-PER`, `Jane=I-PER` |
| **B-ORG** | Begin - Organization | First word of an organization | "Google" → `B-ORG` |
| **I-ORG** | Inside - Organization | Continuation of organization name | "New York Times" → `New=B-ORG`, `York=I-ORG`, `Times=I-ORG` |
| **B-LOC** | Begin - Location | First word of a location | "London" → `B-LOC` |
| **I-LOC** | Inside - Location | Continuation of location name | "New York" → `New=B-LOC`, `York=I-LOC` |
| **B-MISC** | Begin - Miscellaneous | First word of misc entity (event, nationality) | "Indian" → `B-MISC` |
| **I-MISC** | Inside - Miscellaneous | Continuation of misc entity | "South Korean" → `South=B-MISC`, `Korean=I-MISC` |

</div>

<div style="background-color: #ede7f6; padding: 15px; border-radius: 8px;">

### BIO Scheme Explained

- **B** → **Begin** — First word of an entity
- **I** → **Inside** — Continuation words of the same entity
- **O** → **Outside** — Not part of any entity

**Why BIO?** Without B/I distinction, the model can't tell where one entity ends and another begins:
```
"John Smith met Steve Jobs"
With BIO:   B-PER I-PER O B-PER I-PER   ← Two separate persons!
Without B:  PER   PER   O  PER   PER    ← Is it 1 person or 2? Ambiguous!
```

**Dataset used:** WikiAnn (Wikipedia + Annotation) — a multilingual NER dataset

</div>

<div style="background-color: #e8eaf6; padding: 15px; border-left: 5px solid #3f51b5; border-radius: 8px;">

### NER Data Loading Flow

```
WikiAnn Dataset (Hugging Face)
        │
        ▼
 Random sample: 1000 train, 250 test
        │
        ▼
 Extract: tokens list + ner_tags list
        │
        ▼
 Return → Ready for NERDataset & DataLoader
```

**What `load_wikiann_data()` does:**
1. Loads WikiAnn English dataset from Hugging Face
2. Randomly selects training (1000) and test (250) samples
3. Extracts tokens and their NER tags into separate lists
4. Returns them for the training pipeline

</div>

In [ ]:
class NamedEntityModel:
    """End-to-end BERT pipeline for Named Entity Recognition"""

    def __init__(self, base='bert-base-uncased', num_labels=9, seq_len=512):
        self.base = base
        self.num_labels = num_labels
        self.seq_len = seq_len
        self.tok = BertTokenizerFast.from_pretrained(base)
        self.net = BertForTokenClassification.from_pretrained(base, num_labels=num_labels)
        self.net.to(device)
        self.tag_names = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG',
                          'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

    def fetch_wikiann(self, n=1000):
        """Sample n sentences from the WikiAnn English NER dataset"""
        print("Loading WikiAnn NER dataset...")
        ds = load_dataset("wikiann", "en")

        tr_idx = np.random.choice(len(ds['train']), min(n, len(ds['train'])), replace=False)
        te_idx = np.random.choice(len(ds['test']), min(n // 4, len(ds['test'])), replace=False)

        tr_toks = [ds['train'][int(i)]['tokens'] for i in tr_idx]
        tr_tags = [ds['train'][int(i)]['ner_tags'] for i in tr_idx]
        te_toks = [ds['test'][int(i)]['tokens'] for i in te_idx]
        te_tags = [ds['test'][int(i)]['ner_tags'] for i in te_idx]

        print(f"Train: {len(tr_toks)}  |  Test: {len(te_toks)}")
        return tr_toks, tr_tags, te_toks, te_tags

    def fit(self, tr_toks, tr_tags, epochs=1, batch_sz=8, lr=2e-5):
        """Run the NER training loop"""
        train_ds = TagDataset(tr_toks, tr_tags, self.tok, self.seq_len)
        loader = DataLoader(train_ds, batch_size=batch_sz, shuffle=True)

        opt = AdamW(self.net.parameters(), lr=lr, weight_decay=0.01)
        n_steps = len(loader) * epochs
        sched = get_linear_schedule_with_warmup(opt, num_warmup_steps=n_steps // 10,
                                                num_training_steps=n_steps)
        self.net.train()

        for ep in range(epochs):
            running_loss = 0
            bar = tqdm(loader, desc=f'Epoch {ep+1}/{epochs}')
            for batch in bar:
                opt.zero_grad()
                ids = batch['input_ids'].to(device)
                mask = batch['attention_mask'].to(device)
                lbls = batch['labels'].to(device)

                out = self.net(input_ids=ids, attention_mask=mask, labels=lbls)
                loss = out.loss
                running_loss += loss.item()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.net.parameters(), 1.0)
                opt.step()
                sched.step()
                bar.set_postfix({'loss': f'{loss.item():.4f}'})

            print(f'Epoch {ep+1} — avg loss: {running_loss / len(loader):.4f}')

    def score(self, te_toks, te_tags, batch_sz=8):
        """Evaluate token-level accuracy and F1 (ignore -100 tokens)"""
        test_ds = TagDataset(te_toks, te_tags, self.tok, self.seq_len)
        loader = DataLoader(test_ds, batch_size=batch_sz, shuffle=False)

        self.net.eval()
        all_preds, all_true = [], []

        with torch.no_grad():
            for batch in tqdm(loader, desc='Evaluating'):
                ids = batch['input_ids'].to(device)
                mask = batch['attention_mask'].to(device)
                lbls = batch['labels'].to(device)

                out = self.net(input_ids=ids, attention_mask=mask)
                preds = torch.argmax(out.logits, dim=2).cpu().numpy()
                lbls = lbls.cpu().numpy()

                for i in range(preds.shape[0]):
                    for j in range(preds.shape[1]):
                        if lbls[i][j] != -100:
                            all_preds.append(preds[i][j])
                            all_true.append(lbls[i][j])

        acc = accuracy_score(all_true, all_preds)
        f1 = f1_score(all_true, all_preds, average='weighted')
        return acc, f1

    def tag(self, word_seqs):
        """Predict NER tags for a list of tokenised sentences"""
        self.net.eval()
        results = []

        for words in word_seqs:
            enc = self.tok(words, truncation=True, padding='max_length',
                           max_length=self.seq_len, return_tensors='pt',
                           is_split_into_words=True)
            ids = enc['input_ids'].to(device)
            mask = enc['attention_mask'].to(device)

            with torch.no_grad():
                out = self.net(input_ids=ids, attention_mask=mask)
                preds = torch.argmax(out.logits, dim=2).cpu().numpy()[0]

            word_ids = enc.word_ids(batch_index=0)
            word_preds = []
            prev_wid = None
            for k, wid in enumerate(word_ids):
                if wid is not None and wid != prev_wid and wid < len(words):
                    word_preds.append(self.tag_names[preds[k]])
                prev_wid = wid

            results.append(word_preds)

        return results

In [ ]:
class SQuADDataset(Dataset):
    """Dataset for extractive Question Answering (SQuAD format)"""

    def __init__(self, questions, passages, answers, tok, seq_len=512):
        self.questions = questions
        self.passages = passages
        self.answers = answers
        self.tok = tok
        self.seq_len = seq_len

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, pos):
        q = self.questions[pos]
        ctx = self.passages[pos]
        ans = self.answers[pos]

        enc = self.tok(
            q,
            ctx,
            truncation=True,
            padding='max_length',
            max_length=self.seq_len,
            return_offsets_mapping=True,
            return_tensors='pt'
        )

        offsets = enc.pop("offset_mapping")[0]
        start_pos = torch.tensor(0, dtype=torch.long)
        end_pos = torch.tensor(0, dtype=torch.long)

        if ans and 'answer_start' in ans and ans['answer_start']:
            char_start = ans['answer_start'][0]
            span_text = ans['text'][0]
            char_end = char_start + len(span_text)

            for i, (s, e) in enumerate(offsets):
                if s <= char_start < e:
                    start_pos = torch.tensor(i, dtype=torch.long)
                if s < char_end <= e:
                    end_pos = torch.tensor(i, dtype=torch.long)
                    break

        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'start_positions': start_pos,
            'end_positions': end_pos
        }

In [ ]:
class QuestionAnswerModel:
    """End-to-end BERT pipeline for Extractive Question Answering (SQuAD)"""

    def __init__(self, base='bert-base-uncased', seq_len=512):
        self.base = base
        self.seq_len = seq_len
        self.tok = BertTokenizerFast.from_pretrained(base)
        self.net = BertForQuestionAnswering.from_pretrained(base)
        self.net.to(device)

    def fetch_squad(self, n=2000):
        """Sample n QA pairs from SQuAD"""
        print("Loading SQuAD dataset...")
        ds = load_dataset("squad")

        tr_idx = np.random.choice(len(ds['train']), min(n, len(ds['train'])), replace=False)
        val_idx = np.random.choice(len(ds['validation']), min(n // 4, len(ds['validation'])), replace=False)

        tr_q = [ds['train'][int(i)]['question'] for i in tr_idx]
        tr_ctx = [ds['train'][int(i)]['context'] for i in tr_idx]
        tr_ans = [ds['train'][int(i)]['answers'] for i in tr_idx]

        val_q = [ds['validation'][int(i)]['question'] for i in val_idx]
        val_ctx = [ds['validation'][int(i)]['context'] for i in val_idx]
        val_ans = [ds['validation'][int(i)]['answers'] for i in val_idx]

        print(f"Train: {len(tr_q)}  |  Val: {len(val_q)}")
        return tr_q, tr_ctx, tr_ans, val_q, val_ctx, val_ans

    def fit(self, questions, passages, answers, epochs=1, batch_sz=8, lr=2e-5):
        """Run the QA training loop"""
        train_ds = SQuADDataset(questions, passages, answers, self.tok, self.seq_len)
        loader = DataLoader(train_ds, batch_size=batch_sz, shuffle=True)

        opt = AdamW(self.net.parameters(), lr=lr, weight_decay=0.01)
        n_steps = len(loader) * epochs
        sched = get_linear_schedule_with_warmup(opt, num_warmup_steps=n_steps // 10,
                                                num_training_steps=n_steps)
        self.net.train()

        for ep in range(epochs):
            running_loss = 0
            bar = tqdm(loader, desc=f'Epoch {ep+1}/{epochs}')
            for batch in bar:
                opt.zero_grad()
                ids = batch['input_ids'].to(device)
                mask = batch['attention_mask'].to(device)
                start_pos = batch['start_positions'].to(device)
                end_pos = batch['end_positions'].to(device)

                out = self.net(input_ids=ids, attention_mask=mask,
                               start_positions=start_pos, end_positions=end_pos)
                loss = out.loss
                running_loss += loss.item()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.net.parameters(), 1.0)
                opt.step()
                sched.step()
                bar.set_postfix({'loss': f'{loss.item():.4f}'})

            print(f'Epoch {ep+1} — avg loss: {running_loss / len(loader):.4f}')

    def get_answer(self, question, passage, max_span=30):
        """Extract the answer span from a passage given a question"""
        enc = self.tok(question, passage, truncation=True, padding='max_length',
                       max_length=self.seq_len, return_tensors='pt')
        ids = enc['input_ids'].to(device)
        mask = enc['attention_mask'].to(device)

        self.net.eval()
        with torch.no_grad():
            out = self.net(input_ids=ids, attention_mask=mask)
            s_idx = torch.argmax(out.start_logits, dim=1).item()
            e_idx = torch.argmax(out.end_logits, dim=1).item()

            if e_idx < s_idx:
                e_idx = s_idx
            if (e_idx - s_idx) > max_span:
                e_idx = s_idx + max_span

            span_tokens = ids[0][s_idx:e_idx + 1]
            return self.tok.decode(span_tokens, skip_special_tokens=True).strip()

In [ ]:
def launch_sentiment_demo():
    print("\n" + "="*60)
    print("SENTIMENT ANALYSIS (Text Classification) DEMO")
    print("="*60)

    sentiment_model = TextSentimentModel(num_classes=2)
    tr_texts, tr_labels, te_texts, te_labels = sentiment_model.fetch_imdb(n=1000)

    print(f"\nSample review: {tr_texts[0][:200]}...")
    print(f"Label: {'Positive' if tr_labels[0] == 1 else 'Negative'}")

    sentiment_model.fit(tr_texts, tr_labels, epochs=1, batch_sz=8)

    acc, f1, report = sentiment_model.score(te_texts, te_labels, batch_sz=8)
    print(f"\nAccuracy : {acc:.4f}")
    print(f"F1 Score : {f1:.4f}")

    sample_reviews = [
        "Absolutely stunning — the performances were extraordinary!",
        "Dull and predictable. I nearly fell asleep.",
        "It was okay, nothing special but watchable."
    ]

    preds, probs = sentiment_model.infer(sample_reviews)
    print("\nCustom Predictions:")
    for txt, pred, prob in zip(sample_reviews, preds, probs):
        label = "Positive" if pred == 1 else "Negative"
        conf = prob[pred] * 100
        print(f"  '{txt[:55]}...' → {label} ({conf:.1f}%)")

In [ ]:
def launch_ner_demo():
    print("\n" + "="*60)
    print("NAMED ENTITY RECOGNITION DEMO")
    print("="*60)

    ner_model = NamedEntityModel(num_labels=9)
    tr_toks, tr_tags, te_toks, te_tags = ner_model.fetch_wikiann(n=500)

    print(f"\nSample words : {tr_toks[0][:10]}")
    tag_preview = [ner_model.tag_names[t] if t < len(ner_model.tag_names) else "O"
                   for t in tr_tags[0][:10]]
    print(f"Sample tags  : {tag_preview}")

    ner_model.fit(tr_toks, tr_tags, epochs=1, batch_sz=8)

    acc, f1 = ner_model.score(te_toks, te_tags, batch_sz=8)
    print(f"\nAccuracy : {acc:.4f}")
    print(f"F1 Score : {f1:.4f}")

    custom_sents = [
        ["John", "Smith", "works", "at", "Google", "in", "California"],
        ["Apple", "Inc.", "was", "founded", "by", "Steve", "Jobs"]
    ]

    tagged = ner_model.tag(custom_sents)
    print("\nCustom Predictions:")
    for words, tags in zip(custom_sents, tagged):
        print(f"  Words : {words}")
        print(f"  Tags  : {tags}\n")

In [ ]:
def launch_qa_demo():
    print("\n" + "="*60)
    print("QUESTION ANSWERING DEMO")
    print("="*60)

    qa_model = QuestionAnswerModel()
    tr_q, tr_ctx, tr_ans, val_q, val_ctx, val_ans = qa_model.fetch_squad(n=500)

    ans_preview = tr_ans[0]['text'][0] if tr_ans[0]['text'] else 'N/A'
    print(f"\nSample Q : {tr_q[0]}")
    print(f"Context  : {tr_ctx[0][:200]}...")
    print(f"Answer   : {ans_preview}")

    qa_model.fit(tr_q, tr_ctx, tr_ans, epochs=1, batch_sz=4)

    print("\nCustom Q&A Examples:")
    qa_pairs = [
        {
            "question": "What is the capital of France?",
            "passage": "France is a country in Western Europe. Paris is the capital and largest city of France, known for the Eiffel Tower and the Louvre Museum."
        },
        {
            "question": "Who founded Apple?",
            "passage": "Apple Inc. is an American technology company founded by Steve Jobs, Steve Wozniak, and Ronald Wayne in 1976. It is known for products like the iPhone and Mac."
        }
    ]

    for pair in qa_pairs:
        ans = qa_model.get_answer(pair["question"], pair["passage"], max_span=30)
        print(f"  Q: {pair['question']}")
        print(f"  A: {ans}\n")

In [ ]:
print("BERT Multi-Task Trainer")
print("Select a task:")
print("  1 — Sentiment Analysis")
print("  2 — Named Entity Recognition")
print("  3 — Question Answering")
print("  4 — Run All Three")

user_choice = input("\nYour choice (1-4): ").strip()

try:
    if user_choice == "1":
        launch_sentiment_demo()
    elif user_choice == "2":
        launch_ner_demo()
    elif user_choice == "3":
        launch_qa_demo()
    elif user_choice == "4":
        launch_sentiment_demo()
        launch_ner_demo()
        launch_qa_demo()
    else:
        print("Invalid input — please re-run and enter 1, 2, 3, or 4.")
except Exception as err:
    print("\n--- ERROR ---")
    print(f"{err}")
    print("\nChecklist:")
    print("  1. Install packages: pip install torch transformers datasets scikit-learn tqdm numpy")
    print("  2. Run all class definition cells before this cell")
    print("  3. Confirm GPU/CPU device is set correctly")